# 1) Chargement du modèle

In [1]:
import pandas as pd
import joblib

#Seuil de classification utilisé lors de l'optimisation du modèle
CLASSIFICATION_THRESHOLD = 0.8

#Charger le pipeline pré-entraîné
try:
    pipeline_loaded = joblib.load('pipeline.joblib')
    print("✅ Pipeline chargé avec succès.")
except FileNotFoundError:
    print("❌ Erreur : Le fichier 'pipeline.joblib' est introuvable. Veuillez le télécharger dans votre environnement Colab.")
    exit()
except Exception as e:
    print(f"❌ Erreur lors du chargement du pipeline : {e}")
    exit()

✅ Pipeline chargé avec succès.


#2) Chargement du fichier

In [2]:
import os


DATA_PATH = "/content/billets_production.csv"

if not os.path.exists(DATA_PATH):
    print(f"❌ Attention : le fichier '{DATA_PATH}' n'existe pas.")

try:
    df = pd.read_csv(DATA_PATH,delimiter=',')
    print("✅ Données chargées :", df.shape)
    display(df)
except Exception as e:
    print("❌ Erreur lors du chargement :", e)

✅ Données chargées : (5, 7)


,diagonal,height_left,height_right,margin_low,margin_up,length,id
0,171.76,104.01,103.54,5.21,3.30,111.42,A_1
1,171.87,104.17,104.13,6.00,3.31,112.09,A_2
2,172.00,104.58,104.29,4.99,3.39,111.57,A_3
3,172.49,104.55,104.34,4.44,3.03,113.20,A_4
4,171.65,103.63,103.56,3.77,3.16,113.33,A_5


#3) Application de l'algorithme

In [3]:
#Vérifier les colonnes manquantes dans les nouvelles données
df_nan = df.loc[df.isna().any(axis=1)]
df_clean = df.dropna().copy()

In [4]:
import numpy as np

df_predictions_results = pd.DataFrame(columns=['predicted_genuine', 'probability_genuine'])

if not df_clean.empty:
    #Identifier les colonnes numériques pertinentes pour le modèle
    model_features = ['height_left', 'height_right', 'margin_low', 'margin_up', 'length']

    #Filtrer df_clean pour ne garder que les colonnes attendues par le modèle
    X_predict = df_clean[model_features]

    #Faire des prédictions sur les billets sans valeurs manquantes
    probabilities = pipeline_loaded.predict_proba(X_predict)[:, 1]

    predictions = (probabilities >= CLASSIFICATION_THRESHOLD).astype(bool)

    # Assignation directe des résultats aux nouvelles colonnes dans df_clean
    df_predictions_results = pd.DataFrame({
    'predicted_genuine': predictions,
    'probability_genuine': probabilities
})

    df_clean = df_clean.reset_index(drop=True).join(df_predictions_results)

#4) Résultats

In [7]:
print("\n--- Résultats des prédictions pour les billets sans valeurs manquantes ---")
if not df_clean.empty:
    display(df_clean[['id','predicted_genuine']])
else:
    print("Aucun billet sans valeur manquante n'a été trouvé pour la prédiction.")

print("\n--- Billets non traités en raison de valeurs manquantes ---")
if not df_nan.empty:
    display(df_nan['id'])
    print("Ces billets n'ont pas été traités car ils contiennent des valeurs manquantes.")
else:
    print("Tous les billets ont pu être traités (aucune valeur manquante).")


--- Résultats des prédictions pour les billets sans valeurs manquantes ---


,id,predicted_genuine
0,A_1,False
1,A_2,False
2,A_3,False
3,A_4,True
4,A_5,True



--- Billets non traités en raison de valeurs manquantes ---
Tous les billets ont pu être traités (aucune valeur manquante).
